In [22]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed").exists())

# ก่อน prepare: ป้ายดิบตามที่กรอกใน Excel
raw = pd.read_csv(ROOT / "processed/normalize/properties.csv", dtype=str)
raw = raw[raw["property"].str.contains("grain", case=False, na=False)]

print("ป้าย grain size ก่อน prepare (ตามที่กรอกใน Excel)")
for label, g in raw.groupby("property"):
    print(f"  {label:40} {len(g):3} แถว · {g['value'].notna().sum():3} แถวมีค่ากรอก")
print(f"  รวม {raw['property'].nunique()} แบบ")

# หลัง prepare: ป้ายที่ EDA เห็น
prep = pd.read_csv(ROOT / "processed/prepared_v2/properties.csv")
prep = prep[prep["property"].str.contains("grain", case=False, na=False)]

print("\nหลัง prepare (base_property / layer)")
for (base, layer), g in prep.groupby(["base_property", prep["layer"].fillna("-")]):
    print(f"  {base:40} layer={layer:4} {len(g):3} แถว · {g['value_parsed'].notna().sum():3} แถวมีตัวเลข · ชื่อดิบ: {sorted(g['property'].unique())}")
print(f"  รวม {prep['base_property'].nunique()} base_property")


ป้าย grain size ก่อน prepare (ตามที่กรอกใน Excel)
  Grain Size                                28 แถว ·  13 แถวมีค่ากรอก
  Grain Size (Bottom)                        3 แถว ·   3 แถวมีค่ากรอก
  Grain Size (Top)                           3 แถว ·   3 แถวมีค่ากรอก
  Grain size                                10 แถว ·   8 แถวมีค่ากรอก
  Grain size range                           1 แถว ·   0 แถวมีค่ากรอก
  HTL grain size                             3 แถว ·   3 แถวมีค่ากรอก
  PSK grain size                             3 แถว ·   3 แถวมีค่ากรอก
  Perovskite Grain Size                      2 แถว ·   2 แถวมีค่ากรอก
  Perovskite Grain Size (cross-section)      2 แถว ·   2 แถวมีค่ากรอก
  Perovskite Grain Size (top-view)           2 แถว ·   2 แถวมีค่ากรอก
  รวม 10 แบบ

หลัง prepare (base_property / layer)
  Grain Size (Bottom)                      layer=-      3 แถว ·   3 แถวมีตัวเลข · ชื่อดิบ: ['Grain Size (Bottom)']
  Grain Size (Top)                         layer=-      3 แถว ·   3 แถวมีตัวเลข · ชื

In [21]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed").exists())

# หลัง prepare: ตาราง properties + devices ใน prepared_v2
prep = pd.read_csv(ROOT / "processed/prepared_v2/properties.csv")
dev = pd.read_csv(ROOT / "processed/prepared_v2/devices.csv", usecols=["device_id", "doi_raw", "source_file"])

g = prep[prep["property"].str.contains("grain", case=False, na=False)].merge(dev, on="device_id", how="left")


def item(r):
    label = r["base_property"] + (f" [layer={r['layer']}]" if pd.notna(r["layer"]) else "")
    val = f"{r['value_parsed']:g}" if pd.notna(r["value_parsed"]) else "(ว่าง)"
    unit = r["unit_corrected"] if pd.notna(r["unit_corrected"]) else ""
    text = f"{label} = {val} {unit}".strip()
    # ถ้าค่าดิบที่กรอกไม่ใช่ตัวเลขล้วน (เช่น ">200", "100-200") ให้แสดงไว้ข้างหลัง จะได้เห็นว่า prepare แปลงอะไร
    raw = str(r["value"]) if pd.notna(r["value"]) else ""
    try:
        float(raw)
    except ValueError:
        if raw:
            text += f"   (ดิบ: {raw})"
    return text


g["item"] = g.apply(item, axis=1)

# 1 แถว = 1 ไฟล์ Excel (1 ไฟล์ = 1 เครื่อง)
files = g.groupby(["doi_raw", "source_file"])["item"].agg("; ".join).reset_index()

for doi, x in files.groupby("doi_raw"):
    print(doi)
    for r in x.itertuples():
        print(f"   {r.source_file:45} {r.item}")

print(f"\n{files['doi_raw'].nunique()} paper · {len(files)} ไฟล์")

10.1002_smtd.202200787
   10.1002_smtd.202200787_Sample1.xlsx           Grain size = 150 nm
   10.1002_smtd.202200787_Sample2.xlsx           Grain size = 200 nm   (ดิบ: >200)
10.1002_solr.202400534
   10.1002_solr.202400534_Sample1.xlsx           Perovskite Grain Size = 0.69 µm   (ดิบ: 0.18-1.2)
   10.1002_solr.202400534_Sample2.xlsx           Perovskite Grain Size = 0.885 µm   (ดิบ: 0.34-1.43)
10.1002_solr.202500173
   10.1002_solr.202500173_Sample1.xlsx           Grain size = 257 nm
   10.1002_solr.202500173_Sample2.xlsx           Grain size = 302 nm
10.1016_j.solener.2024.112888
   10.1016_j.solener.2024.112888_Sample1.xlsx    Grain size = 220 nm
   10.1016_j.solener.2024.112888_Sample2.xlsx    Grain size = 310.5 nm
10.1016j.jechem.2021.11.029
   10.1016j.jechem.2021.11.029_Sample1.xlsx      Grain size = 283 nm
   10.1016j.jechem.2021.11.029_Sample2.xlsx      Grain size = 329 nm
10.1021_acsami.3c02156
   10.1021_acsami.3c02156_Sample1.xlsx           Grain size = (ว่าง)
   10.1021_ac